In [9]:
import pandas as pd
from google.cloud import storage
import tempfile, os
from IPython.display import display

In [10]:
from google.cloud import storage

GCP_PROJECT = "moeyens-thor-dev"
storage_client = storage.Client(project=GCP_PROJECT)

/Users/b612foundation/b612_packages/adam-thor-research/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [11]:
# Local path to the downloaded analysis_observations.parquet file
analysis_observations_file = "/Users/b612foundation/b612_packages/data/analysis_observations.parquet"

In [13]:
# First, get the analysis observations we need from the local parquet file.
# This file is large, so only load the columns needed for labeling transformed detections.

#analysis_observations_df = pd.read_parquet(
#    analysis_observations_file,``
#    columns=["id", "object_id"],  # orbit id / None for noise
#)

#analysis_observations_df.head()


In [14]:
#write this to a parquet file for faster loading  in the future 
#analysis_observations_df.to_parquet("analysis_observations_small.parquet")

analysis_observations_df = pd.read_parquet("analysis_observations_small.parquet")

In [17]:
import numpy as np
import pandas as pd

def compute_linearity(group: pd.DataFrame) -> pd.Series:
    # unpack nested coordinates into plain columns
    coords = pd.json_normalize(group["coordinates"])
    x = coords["theta_x"].to_numpy()
    y = coords["theta_y"].to_numpy()
    n = len(x)
    if n < 2:
        return pd.Series({"n_obs": n, "linearity_r2": np.nan})

    # least-squares line fit: y ≈ m x + b
    m, b = np.polyfit(x, y, 1)
    y_hat = m * x + b

    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 1.0

    return pd.Series({"n_obs": n, "linearity_r2": r2})

In [19]:
# GCS configuration
GCP_PROJECT = "moeyens-thor-dev"
BUCKET = "thor-research"
BASE_PREFIX = "runs/dec3"

# Analysis configuration
MIN_OBS_PER_ORBIT = 6  # stop when we find any object with >= 6 observations

storage_client = storage.Client(project=GCP_PROJECT)
bucket = storage_client.bucket(BUCKET)

# Find all transformed_detections.parquet files for the test orbits
blobs = storage_client.list_blobs(BUCKET, prefix=f"{BASE_PREFIX}/")
orbit_parquet_paths = sorted(
    [blob.name for blob in blobs if blob.name.endswith("transformed_detections.parquet")]
)

print(f"Found {len(orbit_parquet_paths)} candidate orbit files.")

recovered_observations_df = None
orbits_with_enough_obs = None
selected_orbit_path = None

for blob_name in orbit_parquet_paths:
    gcs_uri = f"gs://{BUCKET}/{blob_name}"
    print(f"Checking {gcs_uri} ...")

    # Download this orbit's transformed detections to a temporary file and load with pandas
    blob = bucket.blob(blob_name)
    with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as tmp:
        blob.download_to_filename(tmp.name)
        tmp_path = tmp.name

    df = pd.read_parquet(tmp_path)
    os.remove(tmp_path)

    #print(df.head())

    df_with_labels = df.merge(analysis_observations_df, on="id", how="left")

    #print(df_with_labels.head())

    #now remove the none in the object_id column 
    print(f"DF size before filtering: {df_with_labels.shape}")
    df_with_labels = df_with_labels[df_with_labels['object_id'].notna()]
    print(f"DF size after filtering: {df_with_labels.shape}")

    #now group by object_id and filter for the ones with more than 6 observations 
    grouped_by_orbit = df_with_labels.groupby("object_id")
    print(grouped_by_orbit.head())
    orbits_with_enough_obs = grouped_by_orbit.filter(lambda g: len(g) >= MIN_OBS_PER_ORBIT)

    #print size of enough 
    print(f"Enough size: {orbits_with_enough_obs.shape}")

    #print each object_id in orbits_with_enough_obs and number of observations it has 
    for object_id, count in orbits_with_enough_obs["object_id"].value_counts().items():
        print(f"Object ID: {object_id}, Count: {count}")

    summary = (
        orbits_with_enough_obs
        .groupby("object_id")
        .apply(compute_linearity)
        .reset_index()
    )
    print(summary)
    print(blob_name)

    #summary["test_orbit_id"] = test_orbit_id
    #summary = summary[["test_orbit_id", "object_id", "n_obs", "linearity_r2"]]
    #summary

    break

/Users/b612foundation/b612_packages/adam-thor-research/.venv/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Found 5145 candidate orbit files.
Checking gs://thor-research/runs/dec3/link_test_orbit_40085_r1.349_e0.000_nu105.000_psi-15.000/outputs/40085_r1.349_e0.000_nu105.000_psi-15.000/transformed_detections.parquet ...
DF size before filtering: (41319, 5)
DF size after filtering: (75, 5)
                                     id  night  \
1728   00c0f525099b4821a750d05bf4cf2095  61027   
2033   86e5ba4019b64758935f5f548dbd4c81  61027   
2127   afab8ad8c0964685ae7140210ff553a5  61027   
2410   358429c18bb4412aaf1f889518dc872b  61022   
2428   40805b904ef54e1ab052a691e5e82459  61022   
2508   6cb644785ce74314bd22e62788150017  61022   
2618   b18c981ae3604e1ebeff480f1c74ada3  61022   
3562   311a98d6dfd9455d82d29443b77e2608  61009   
5236   ac3ed06df8cd4adb8455fb8878863719  61021   
6140   2fdf3f9ca7cb478b829802bb19cc78db  61026   
6349   8220827722084254a5a660db5362f3ae  61026   
6488   c01ded4b637d49b390283326cd76e130  61026   
6590   e32e08fe2d9049e1943f6e928a86e551  61026   
7082   0d9816b97f

/var/folders/jn/6gvx57b56bjdzktff3dt1p3r0000gn/T/ipykernel_70195/920704302.py:63: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_linearity)


In [ ]:
#for each test orbit, group observations with the same orbit id  
# keep the objects with more than 6 observations 

MIN_OBS_PER_ORBIT = 6  

# Keep only orbits with more than 6 observations
orbits_with_enough_obs = (
    recovered_observations_df
    .groupby("orbit_id")
    .filter(lambda g: len(g) >= MIN_OBS_PER_ORBIT)
)

print(
    f"{orbits_with_enough_obs['orbit_id'].nunique()} orbits with ≥{MIN_OBS_PER_ORBIT} observations "
    f"out of {recovered_observations_df['orbit_id'].nunique()} total orbits."
)

# Optional: get a dict from orbit_id -> DataFrame of its observations
orbits_by_id = {
    orbit_id: df for orbit_id, df in orbits_with_enough_obs.groupby("orbit_id")
}

In [ ]:
# for each of these, determine how linear the observations are 